# ReviewRadar — Layer 3: Exploratory Data Analysis

**Input:** `data/processed/reviews_clean.csv`

This notebook explores the cleaned reviews **before any modeling**. Goal: understand rating patterns, review volume trends, and version-wise quality — and surface real, data-backed findings.

> Every 'Initial Finding' at the bottom is **computed from the data at runtime**, not hardcoded.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.titleweight'] = 'bold'

df = pd.read_csv('../data/processed/reviews_clean.csv')
df['review_datetime'] = pd.to_datetime(df['review_datetime'], errors='coerce')
df['reviewCreatedVersion'] = df['reviewCreatedVersion'].fillna('unknown')
print(f'Loaded {len(df):,} reviews')
df.head(3)

## 1. Overall rating distribution
**What it shows:** how many reviews fall at each star level (1–5).
**Why it matters:** the shape (J-curve, skew) tells us whether users are broadly happy or polarised.
**Business question:** *Is our overall app perception positive, negative, or split?*

In [ ]:
ax = sns.countplot(data=df, x='score', order=[1,2,3,4,5], color='#4C72B0')
total = len(df)
for p in ax.patches:
    pct = 100 * p.get_height() / total
    ax.annotate(f'{pct:.1f}%', (p.get_x()+p.get_width()/2, p.get_height()),
                ha='center', va='bottom', fontsize=9)
ax.set(title='Overall Rating Distribution', xlabel='Star rating', ylabel='Number of reviews')
plt.tight_layout(); plt.show()

## 2. Review volume over time
**What it shows:** how many reviews arrive each week.
**Why it matters:** spikes often line up with releases, outages, or marketing pushes.
**Business question:** *When did users get unusually vocal, and is engagement growing or fading?*

In [ ]:
weekly = df.set_index('review_datetime').resample('W').size()
ax = weekly.plot(color='#4C72B0')
ax.set(title='Review Volume Over Time (weekly)', xlabel='Week', ylabel='Reviews')
plt.tight_layout(); plt.show()

## 3. Average rating by month, week, and app version
**What it shows:** how satisfaction moves over time and across builds.
**Why it matters:** a dropping trend or a bad version is an early warning of a product problem.
**Business question:** *Is satisfaction improving or declining, and did any release hurt it?*

In [ ]:
monthly_avg = df.set_index('review_datetime')['score'].resample('M').mean()
ax = monthly_avg.plot(marker='o', color='#55A868')
ax.set(title='Average Rating by Month', xlabel='Month', ylabel='Avg rating', ylim=(1,5))
plt.tight_layout(); plt.show()

# Weekly view only if we have enough distinct weeks to be meaningful
weekly_avg = df.set_index('review_datetime')['score'].resample('W').mean()
if weekly_avg.notna().sum() >= 8:
    ax = weekly_avg.plot(color='#55A868', alpha=0.8)
    ax.set(title='Average Rating by Week', xlabel='Week', ylabel='Avg rating', ylim=(1,5))
    plt.tight_layout(); plt.show()
else:
    print('Not enough weeks for a weekly rating trend.')

## 4. Star-rating percentage mix over time
**What it shows:** the monthly share of 1–5 star reviews (stacked to 100%).
**Why it matters:** a rising 1-star share is the clearest signal of a growing problem, even if the average looks stable.
**Business question:** *Is the proportion of angry (1-star) users increasing?*

In [ ]:
mix = (df.groupby(['review_month', 'score']).size()
         .groupby(level=0).apply(lambda s: 100*s/s.sum())
         .unstack(fill_value=0))
mix = mix.reindex(columns=[1,2,3,4,5], fill_value=0)
ax = mix.plot(kind='area', stacked=True, colormap='RdYlGn', alpha=0.9)
ax.set(title='Star Rating Mix Over Time (%)', xlabel='Month', ylabel='% of reviews', ylim=(0,100))
ax.legend(title='Stars', bbox_to_anchor=(1.01,1))
plt.tight_layout(); plt.show()

## 5. Review volume & average rating by app version
**What it shows:** which versions drew the most reviews and how they were rated.
**Why it matters:** ties user sentiment to specific releases — the core of version analysis later.
**Business question:** *Which release was best/worst received?*

In [ ]:
# Only versions with enough reviews to be statistically meaningful
MIN_REVIEWS = 30
vc = df['reviewCreatedVersion'].value_counts()
keep = vc[(vc.index != 'unknown') & (vc >= MIN_REVIEWS)].index
vdf = df[df['reviewCreatedVersion'].isin(keep)]

vol = vdf['reviewCreatedVersion'].value_counts().sort_values(ascending=False).head(15)
ax = vol.plot(kind='bar', color='#4C72B0')
ax.set(title=f'Review Count by App Version (>= {MIN_REVIEWS} reviews)', xlabel='Version', ylabel='Reviews')
plt.tight_layout(); plt.show()

avg_ver = vdf.groupby('reviewCreatedVersion')['score'].mean().loc[vol.index]
ax = avg_ver.plot(kind='bar', color=np.where(avg_ver >= df['score'].mean(), '#55A868', '#C44E52'))
ax.axhline(df['score'].mean(), ls='--', color='grey', label='overall avg')
ax.set(title='Average Rating by App Version', xlabel='Version', ylabel='Avg rating', ylim=(1,5))
ax.legend(); plt.tight_layout(); plt.show()

## 6. Negative (1-star) review percentage over time
**What it shows:** the monthly share of 1-star reviews on its own.
**Why it matters:** isolates the anger signal that a blended average can hide.
**Business question:** *When did user frustration peak?*

In [ ]:
one_star_pct = df.groupby('review_month').apply(lambda g: 100*(g['score']==1).mean())
ax = one_star_pct.plot(marker='o', color='#C44E52')
ax.set(title='1-Star Review Percentage Over Time', xlabel='Month', ylabel='% 1-star reviews')
plt.tight_layout(); plt.show()

## 7. Anomaly detection — spikes & drops
**What it shows:** weeks where volume, average rating, or 1-star share deviate sharply (|z-score| > 2) from the norm.
**Why it matters:** anomalies are where the interesting product stories live (outages, bad releases).
**Business question:** *Which specific weeks need investigation?*

In [ ]:
def flag_anomalies(series, label):
    s = series.dropna()
    if len(s) < 8 or s.std() == 0:
        print(f'{label}: not enough data to flag anomalies.'); return
    z = (s - s.mean()) / s.std()
    hits = z[z.abs() > 2]
    if hits.empty:
        print(f'{label}: no strong anomalies (|z|>2).')
    else:
        print(f'{label}: {len(hits)} anomalous week(s):')
        for wk, val in s.loc[hits.index].items():
            print(f'   {wk.date()}  value={val:.2f}  (z={z.loc[wk]:+.1f})')

wk_vol = df.set_index('review_datetime').resample('W').size()
wk_rating = df.set_index('review_datetime')['score'].resample('W').mean()
wk_1star = df.set_index('review_datetime')['score'].resample('W').apply(lambda s: 100*(s==1).mean())
flag_anomalies(wk_vol, 'Review volume')
flag_anomalies(wk_rating, 'Average rating')
flag_anomalies(wk_1star, '1-star percentage')

## Initial Findings (computed from the data)
The cell below derives every observation from the actual dataset — nothing is hardcoded.

In [ ]:
f = []
n = len(df)
avg = df['score'].mean()
pct = lambda mask: 100*mask.mean()

f.append(f'Dataset covers {n:,} reviews from {df["review_date"].min()} to {df["review_date"].max()}.')
f.append(f'Overall average rating is {avg:.2f}/5.')
f.append(f'{pct(df["score"]==5):.1f}% of reviews are 5-star and {pct(df["score"]==1):.1f}% are 1-star (polarisation check).')
f.append(f'Sentiment split: {pct(df["rating_category"]=="Positive"):.1f}% Positive, {pct(df["rating_category"]=="Neutral"):.1f}% Neutral, {pct(df["rating_category"]=="Negative"):.1f}% Negative.')

best_m = monthly_avg.idxmax(); worst_m = monthly_avg.idxmin()
f.append(f'Best-rated month: {best_m.date()} ({monthly_avg.max():.2f}); worst-rated: {worst_m.date()} ({monthly_avg.min():.2f}).')

peak_wk = wk_vol.idxmax()
f.append(f'Highest review volume week started {peak_wk.date()} with {int(wk_vol.max())} reviews.')

if len(keep) > 0:
    best_v = avg_ver.idxmax(); worst_v = avg_ver.idxmin()
    f.append(f'Among versions with >={MIN_REVIEWS} reviews, best-rated is {best_v} ({avg_ver.max():.2f}), worst is {worst_v} ({avg_ver.min():.2f}).')

f.append(f'Median review length is {df["review_length"].median():.0f} words; {pct(df["review_length"]<=3):.1f}% are very short (<=3 words).')

first_half = one_star_pct.iloc[:len(one_star_pct)//2].mean()
second_half = one_star_pct.iloc[len(one_star_pct)//2:].mean()
trend = 'risen' if second_half > first_half else 'fallen'
f.append(f'1-star share has {trend}: {first_half:.1f}% (earlier period) vs {second_half:.1f}% (later period).')

print('INITIAL FINDINGS')
print('='*60)
for i, item in enumerate(f, 1):
    print(f'{i}. {item}')